# Notebook 01: Hugging Face Duplication Sweep

## Purpose

This notebook executes, **on Kaggle only**, a deep Hugging Face duplication
sweep to determine whether the benchmark **SLM Efficiency Frontier** is an
exact duplicate, near duplicate, adjacent project, or sufficiently original.

Benchmark context:
> SLM Efficiency Frontier: A PyTorch-first benchmark for measuring when small
> language models beat larger cheap models in cost-per-correct-answer on
> verifiable agentic tasks.

The v0.1 benchmark is strictly English-only.

This notebook is metadata-level research. It does not evaluate models and does
not produce benchmark scores.


## Kaggle-only policy

- This notebook must be executed **only on Kaggle**. Do not run it locally.
- It does **not** call OpenRouter.
- It does **not** perform paid inference.
- It does **not** train models.
- It does **not** download large model weights or large datasets.
- It does **not** use GPU.
- It only queries Hugging Face Hub metadata and, when necessary, small
  README/cards for a limited number of candidate repos.
- It uses simple rate limiting and saves incremental results.


## 1. Environment setup


In [ ]:
# Environment setup (Kaggle only).
import sys
import platform
import subprocess

print("Python:", sys.version)
print("Platform:", platform.platform())

try:
    import torch  # noqa: F401
    print("Torch available (not required for this notebook).")
except Exception as exc:  # pragma: no cover
    print("Torch not required for this notebook:", exc)

# Ensure huggingface_hub is available on Kaggle.
try:
    import huggingface_hub
    print("huggingface_hub:", huggingface_hub.__version__)
except ImportError:
    print("Installing huggingface_hub on Kaggle...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    import huggingface_hub
    print("huggingface_hub installed:", huggingface_hub.__version__)

from huggingface_hub import HfApi
import time
import json
import os
import datetime
from pathlib import Path

print("Setup complete.")


## 2. Project path detection


In [ ]:
# Locate the project root from Kaggle input/working or the local repo layout.
CANDIDATES = [
    Path("/kaggle/input/slm-efficiency-frontier"),
    Path("/kaggle/working/slm-efficiency-frontier"),
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]

PROJECT_ROOT = None
for candidate in CANDIDATES:
    if (candidate / "work" / "1_research").exists() or (candidate / ".core").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()

RESEARCH_ARTIFACTS_DIR = PROJECT_ROOT / "work" / "1_research"
REVIEW_DIR = PROJECT_ROOT / "work" / "4_for-review"
RELEASE_DIR = PROJECT_ROOT / "artifacts/release"

for directory in (RESEARCH_ARTIFACTS_DIR, REVIEW_DIR, RELEASE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = RESEARCH_ARTIFACTS_DIR / "hf_duplication_sweep_results.json"
REPORT_PATH = RESEARCH_ARTIFACTS_DIR / "hf_duplication_sweep.md"
LEGITIMACY_PATH = RESEARCH_ARTIFACTS_DIR / "legitimacy_qa_report.md"
SNAPSHOT_PATH = REVIEW_DIR / "research_snapshot.md"
QA_FINDINGS_PATH = RELEASE_DIR / "qa_findings.md"
RESEARCH_FINDINGS_PATH = RELEASE_DIR / "research_findings.md"
CHANGE_LOG_PATH = RELEASE_DIR / "change_log.md"
OPEN_QUESTIONS_PATH = RELEASE_DIR / "open_questions.md"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESEARCH_ARTIFACTS_DIR:", RESEARCH_ARTIFACTS_DIR)


## 3. Hugging Face Hub client setup

No token is required for public metadata listing. The sweep stays economical:
small page sizes, a sleep between paged requests, and a cap on README fetches.


In [ ]:
api = HfApi()

# Economical research controls.
RATE_LIMIT_SLEEP = 1.0          # seconds between paged requests
MAX_RESULTS_PER_QUERY = 25      # cap per query per hub type
README_FETCH_CAP = 15           # only fetch READMEs for at most this many repos
EXECUTED_AT = datetime.datetime.now(datetime.timezone.utc).isoformat()

print("HfApi ready. Executed at (UTC):", EXECUTED_AT)


## 4. Search query configuration

Queries cover SLM benchmarks, cost-efficiency, OpenRouter, tool calling, JSON
validity, structured extraction, agentic tasks, leaderboards, budget-aware
evaluation, clause conflict, table reasoning, and rerankers.


In [ ]:
QUERIES = [
    "SLM benchmark",
    "small language model benchmark",
    "small model benchmark",
    "cost per correct answer",
    "cost efficiency benchmark",
    "cost aware LLM evaluation",
    "OpenRouter benchmark",
    "tool calling benchmark",
    "JSON validity benchmark",
    "structured extraction benchmark",
    "verifiable agentic benchmark",
    "agentic task benchmark",
    "cheap LLM benchmark",
    "small models as judges",
    "small models tool calling",
    "LLM cost leaderboard",
    "budget aware evaluation",
    "clause conflict benchmark",
    "table reasoning benchmark",
    "reranker benchmark",
]

HUB_TYPES = ["model", "dataset", "space"]

# Feature groups used for duplication scoring. Each group has a weight; weights
# sum to 100. Terms are lowercased and matched as substrings.
BENCHMARK_FEATURES = {
    "theme": {
        "weight": 15,
        "terms": ["slm", "small language model", "efficiency frontier",
                  "cost per correct", "small model"],
    },
    "metric": {
        "weight": 15,
        "terms": ["cost per correct answer", "cost per correct",
                  "cost efficiency", "cost aware", "cost per 1k"],
    },
    "task": {
        "weight": 15,
        "terms": ["tool calling", "tool call", "json validity", "valid json",
                  "structured extraction", "clause conflict", "table reasoning",
                  "reranker", "rerank", "agentic", "safe action"],
    },
    "cost": {
        "weight": 10,
        "terms": ["cost", "budget", "price", "cheap", "spend"],
    },
    "openrouter": {
        "weight": 10,
        "terms": ["openrouter"],
    },
    "slm": {
        "weight": 10,
        "terms": ["slm", "small model", "small language model", "small llm"],
    },
    "leaderboard": {
        "weight": 5,
        "terms": ["leaderboard", "leaderboards", "ranking"],
    },
    "pytorch": {
        "weight": 5,
        "terms": ["pytorch", "torch"],
    },
    "hf_deploy": {
        "weight": 10,
        "terms": ["space", "dataset", "leaderboard", "gradio"],
    },
    "model_pool": {
        "weight": 5,
        "terms": ["model pool", "model selection", "model comparison",
                  "model evaluation"],
    },
}

assert sum(g["weight"] for g in BENCHMARK_FEATURES.values()) == 100

print("Queries:", len(QUERIES))
print("Hub types:", HUB_TYPES)
print("Feature groups:", len(BENCHMARK_FEATURES))


## 5. Models search


In [ ]:
def _common_fields(item, hub_type, query):
    return {
        "hub_type": hub_type,
        "id": getattr(item, "id", None),
        "author": getattr(item, "author", None),
        "likes": getattr(item, "likes", None),
        "downloads": getattr(item, "downloads", None),
        "last_modified": str(getattr(item, "last_modified", "") or ""),
        "tags": list(getattr(item, "tags", []) or []),
        "pipeline_tag": getattr(item, "pipeline_tag", None),
        "matched_query": query,
        "url": "",
    }


def search_models(query, limit=MAX_RESULTS_PER_QUERY):
    results = []
    try:
        for item in api.list_models(search=query, limit=limit):
            record = _common_fields(item, "model", query)
            record["url"] = f"https://huggingface.co/{item.id}"
            results.append(record)
            time.sleep(RATE_LIMIT_SLEEP)
    except Exception as exc:
        print("model search error:", query, exc)
    return results


In [ ]:
all_models = []
for query in QUERIES:
    print("models:", query)
    all_models.extend(search_models(query))
print("raw model hits:", len(all_models))


## 6. Datasets search


In [ ]:
def search_datasets(query, limit=MAX_RESULTS_PER_QUERY):
    results = []
    try:
        for item in api.list_datasets(search=query, limit=limit):
            record = _common_fields(item, "dataset", query)
            record["url"] = f"https://huggingface.co/datasets/{item.id}"
            results.append(record)
            time.sleep(RATE_LIMIT_SLEEP)
    except Exception as exc:
        print("dataset search error:", query, exc)
    return results


In [ ]:
all_datasets = []
for query in QUERIES:
    print("datasets:", query)
    all_datasets.extend(search_datasets(query))
print("raw dataset hits:", len(all_datasets))


## 7. Spaces search


In [ ]:
def search_spaces(query, limit=MAX_RESULTS_PER_QUERY):
    results = []
    try:
        for item in api.list_spaces(search=query, limit=limit):
            record = _common_fields(item, "space", query)
            # Spaces may not expose pipeline_tag/downloads.
            record["pipeline_tag"] = None
            record["downloads"] = getattr(item, "downloads", None)
            record["url"] = f"https://huggingface.co/spaces/{item.id}"
            results.append(record)
            time.sleep(RATE_LIMIT_SLEEP)
    except Exception as exc:
        print("space search error:", query, exc)
    return results


In [ ]:
all_spaces = []
for query in QUERIES:
    print("spaces:", query)
    all_spaces.extend(search_spaces(query))
print("raw space hits:", len(all_spaces))


## 8. Metadata enrichment

Build a short textual summary per repo from id, tags, and pipeline tag. README
fetching is optional and capped to keep the sweep economical. Only a small
number of candidate repos get their README/card text fetched.


In [ ]:
def card_summary(record):
    parts = []
    if record.get("id"):
        parts.append(str(record["id"]).lower())
    if record.get("pipeline_tag"):
        parts.append(str(record["pipeline_tag"]).lower())
    tags = record.get("tags") or []
    parts.extend(str(t).lower() for t in tags)
    return " ".join(parts)


def fetch_readme_text(record, cap_state):
    """Optionally fetch a small README/card for a repo. Capped globally."""
    if cap_state["count"] >= README_FETCH_CAP:
        return ""
    repo_id = record.get("id")
    if not repo_id:
        return ""
    repo_type = record.get("hub_type")
    if repo_type not in ("model", "dataset", "space"):
        return ""
    try:
        from huggingface_hub import hf_hub_download
        filename = "README.md"
        path = hf_hub_download(repo_id=repo_id, filename=filename,
                               repo_type=repo_type)
        with open(path, "r", encoding="utf-8", errors="ignore") as fh:
            text = fh.read()
        cap_state["count"] += 1
        time.sleep(RATE_LIMIT_SLEEP)
        # Keep only the first 2000 chars to stay economical.
        return text[:2000]
    except Exception as exc:
        return ""


def enrich(records, fetch_readmes=False):
    cap_state = {"count": 0}
    for record in records:
        record["card_summary"] = card_summary(record)
        if fetch_readmes:
            record["readme_excerpt"] = fetch_readme_text(record, cap_state)
        else:
            record["readme_excerpt"] = ""
    return records


all_raw = all_models + all_datasets + all_spaces
all_raw = enrich(all_raw, fetch_readmes=True)
print("enriched records:", len(all_raw))


## 9. Similarity and duplication scoring

Each repo is scored 0-100 against the benchmark feature groups. The score and
the relative strength of domain vs metric overlap determine the duplication
category from the taxonomy.


In [ ]:
def matched_terms_for(record):
    text = " ".join([
        record.get("card_summary", ""),
        record.get("readme_excerpt", ""),
        str(record.get("matched_query", "")).lower(),
    ])
    matched = {}
    for group, info in BENCHMARK_FEATURES.items():
        hits = [term for term in info["terms"] if term in text]
        if hits:
            matched[group] = hits
    return matched


def score_record(record):
    matched = matched_terms_for(record)
    score = 0
    for group, info in BENCHMARK_FEATURES.items():
        if group in matched:
            score += info["weight"]

    domain_groups = {"theme", "task", "slm", "hf_deploy"}
    metric_groups = {"metric", "cost"}
    domain_weight = sum(BENCHMARK_FEATURES[g]["weight"] for g in domain_groups if g in matched)
    metric_weight = sum(BENCHMARK_FEATURES[g]["weight"] for g in metric_groups if g in matched)

    if score == 0 and not record.get("card_summary"):
        category = "insufficient_information"
    elif score >= 91:
        category = "exact_duplicate"
    elif score >= 71:
        category = "near_duplicate"
    elif score >= 46:
        if metric_weight > domain_weight:
            category = "same_metric_different_domain"
        else:
            category = "same_domain_different_metric"
    elif score >= 21:
        category = "adjacent_project"
    else:
        category = "not_duplicate"

    reasoning = (
        f"score={score}; matched_groups={sorted(matched.keys())}; "
        f"domain_weight={domain_weight}; metric_weight={metric_weight}; "
        f"hub_type={record.get('hub_type')}"
    )

    actions = {
        "exact_duplicate": "STOP: reposition or redesign the benchmark.",
        "near_duplicate": "REVISE: add a clear differentiation argument or reposition.",
        "same_domain_different_metric": "REVISE: emphasize the unique cost-per-correct metric.",
        "same_metric_different_domain": "REVISE: emphasize the unique verifiable agentic task domain.",
        "adjacent_project": "PROCEED: document adjacency and differentiation.",
        "not_duplicate": "PROCEED: record as distinct.",
        "insufficient_information": "REVIEW: gather more metadata before deciding.",
    }
    return {
        "matched_terms": matched,
        "duplication_risk_score": score,
        "duplication_category": category,
        "reasoning": reasoning,
        "recommended_action": actions[category],
    }


def score_all(records):
    for record in records:
        record.update(score_record(record))
    return records


all_scored = score_all(all_raw)
print("scored records:", len(all_scored))


## 10. Result deduplication

The same repo may appear under multiple queries. Merge by id, union the matched
queries and matched terms, and keep the highest risk score.


In [ ]:
def deduplicate(records):
    merged = {}
    for record in records:
        key = (record.get("hub_type"), record.get("id"))
        if key is None or key[1] is None:
            continue
        if key not in merged:
            merged[key] = dict(record)
            merged[key]["matched_queries"] = [record.get("matched_query")]
            merged[key]["matched_term_groups"] = dict(record.get("matched_terms", {}))
        else:
            existing = merged[key]
            mq = record.get("matched_query")
            if mq and mq not in existing["matched_queries"]:
                existing["matched_queries"].append(mq)
            for group, terms in record.get("matched_terms", {}).items():
                combined = set(existing["matched_term_groups"].get(group, []))
                combined.update(terms)
                existing["matched_term_groups"][group] = sorted(combined)
            if record.get("duplication_risk_score", 0) > existing.get("duplication_risk_score", 0):
                existing["duplication_risk_score"] = record["duplication_risk_score"]
                existing["duplication_category"] = record["duplication_category"]
                existing["reasoning"] = record["reasoning"]
                existing["recommended_action"] = record["recommended_action"]
    for existing in merged.values():
        existing["matched_queries"] = sorted(filter(None, existing.get("matched_queries", [])))
        existing["matched_terms"] = existing.pop("matched_term_groups", {})
    return list(merged.values())


deduped = deduplicate(all_scored)
print("deduplicated records:", len(deduped))


## 11. Risk ranking

Sort by duplication risk score descending. Separate highest-risk possible
duplicates, adjacent projects, and clearly distinct repos.


In [ ]:
def rank(records):
    return sorted(records, key=lambda r: r.get("duplication_risk_score", 0), reverse=True)


ranked = rank(deduped)

high_risk = [r for r in ranked if r.get("duplication_risk_score", 0) >= 71]
meaningful = [r for r in ranked if 46 <= r.get("duplication_risk_score", 0) < 71]
adjacent = [r for r in ranked if 21 <= r.get("duplication_risk_score", 0) < 46]
distinct = [r for r in ranked if r.get("duplication_risk_score", 0) < 21]

print("high risk (>=71):", len(high_risk))
print("meaningful overlap (46-70):", len(meaningful))
print("adjacent (21-45):", len(adjacent))
print("distinct (0-20):", len(distinct))


## 12. Save incremental results

Write the structured JSON results so partial progress is never lost.


In [ ]:
def serialize_records(records):
    safe = []
    for record in records:
        item = {}
        for key, value in record.items():
            if key == "matched_terms":
                item[key] = {g: list(v) for g, v in value.items()}
            elif isinstance(value, (list, tuple)):
                item[key] = list(value)
            elif isinstance(value, (str, int, float, bool)) or value is None:
                item[key] = value
            else:
                item[key] = str(value)
        safe.append(item)
    return safe


output = {
    "executed_at": EXECUTED_AT,
    "kaggle_only": True,
    "openrouter_used": False,
    "queries": QUERIES,
    "hub_types": HUB_TYPES,
    "max_results_per_query": MAX_RESULTS_PER_QUERY,
    "readme_fetch_cap": README_FETCH_CAP,
    "total_raw_hits": len(all_raw),
    "total_deduplicated": len(deduped),
    "counts": {
        "high_risk": len(high_risk),
        "meaningful_overlap": len(meaningful),
        "adjacent": len(adjacent),
        "distinct": len(distinct),
    },
    "records": serialize_records(ranked),
}

with open(RESULTS_PATH, "w", encoding="utf-8") as fh:
    json.dump(output, fh, indent=2, allow_nan=False)
print("Results written to", RESULTS_PATH)


## 13. Markdown report generation

Generate the English markdown report at
`artifacts/research/hf_duplication_sweep.md`.


In [ ]:
def _row(record):
    return (
        record.get("hub_type"),
        record.get("id"),
        record.get("duplication_category"),
        record.get("duplication_risk_score"),
        record.get("matched_queries"),
        record.get("url"),
    )


def build_report():
    lines = []
    lines.append("# Hugging Face Duplication Sweep Report")
    lines.append("")
    lines.append(f"**Date/time of Kaggle execution (UTC):** {EXECUTED_AT}")
    lines.append("")
    lines.append("## Methodology")
    lines.append("")
    lines.append(
        "Metadata-level search of the Hugging Face Hub (models, datasets, "
        "spaces) using multiple queries. Each result is enriched with a short "
        "card summary and, for a capped subset, a small README excerpt. "
        "Results are scored 0-100 against the benchmark feature groups "
        "(theme, metric, task, cost, OpenRouter, SLM, leaderboard, PyTorch, "
        "HF deployment, model pool) and assigned a duplication category from "
        "the taxonomy. Duplicate hits across queries are merged."
    )
    lines.append("")
    lines.append("### Duplication taxonomy")
    lines.append("")
    lines.append("- `exact_duplicate` (91-100)")
    lines.append("- `near_duplicate` (71-90)")
    lines.append("- `same_domain_different_metric` (46-70)")
    lines.append("- `same_metric_different_domain` (46-70)")
    lines.append("- `adjacent_project` (21-45)")
    lines.append("- `not_duplicate` (0-20)")
    lines.append("- `insufficient_information`")
    lines.append("")
    lines.append("## Search scope")
    lines.append("")
    lines.append(f"- Hub types: {', '.join(HUB_TYPES)}")
    lines.append(f"- Max results per query: {MAX_RESULTS_PER_QUERY}")
    lines.append(f"- README fetch cap: {README_FETCH_CAP}")
    lines.append(f"- Total raw hits: {len(all_raw)}")
    lines.append(f"- Total deduplicated: {len(deduped)}")
    lines.append("")
    lines.append("## Query list")
    lines.append("")
    for query in QUERIES:
        lines.append(f"- {query}")
    lines.append("")
    lines.append("## Summary table")
    lines.append("")
    lines.append("| hub_type | id | category | score | matched_queries | url |")
    lines.append("|----------|----|----------|-------|-----------------|-----|")
    for record in ranked[:50]:
        hub, rid, cat, score, mqs, url = _row(record)
        lines.append(
            f"| {hub} | {rid} | {cat} | {score} | "
            f"{'; '.join(mqs or [])} | {url} |"
        )
    lines.append("")
    lines.append("## Highest-risk possible duplicates")
    lines.append("")
    if high_risk:
        for record in high_risk:
            lines.append(f"- **{record.get('id')}** ({record.get('hub_type')}) "
                         f"score={record.get('duplication_risk_score')} "
                         f"category={record.get('duplication_category')}")
            lines.append(f"  - {record.get('reasoning')}")
            lines.append(f"  - action: {record.get('recommended_action')}")
            lines.append(f"  - url: {record.get('url')}")
    else:
        lines.append("None found.")
    lines.append("")
    lines.append("## Meaningful overlap")
    lines.append("")
    if meaningful:
        for record in meaningful:
            lines.append(f"- {record.get('id')} ({record.get('hub_type')}) "
                         f"score={record.get('duplication_risk_score')} "
                         f"category={record.get('duplication_category')}")
    else:
        lines.append("None found.")
    lines.append("")
    lines.append("## Adjacent projects")
    lines.append("")
    if adjacent:
        for record in adjacent[:20]:
            lines.append(f"- {record.get('id')} ({record.get('hub_type')}) "
                         f"score={record.get('duplication_risk_score')}")
    else:
        lines.append("None found.")
    lines.append("")
    lines.append("## Gaps found")
    lines.append("")
    lines.append(
        "- Queries that returned no results are recorded as gaps in coverage."
    )
    lines.append(
        "- The sweep is metadata-level; deep functional equivalence cannot be "
        "fully ruled out from metadata alone."
    )
    lines.append("")
    lines.append("## Originality assessment")
    lines.append("")
    max_score = max((r.get("duplication_risk_score", 0) for r in ranked), default=0)
    if max_score >= 91:
        assessment = "BLOCKED: an exact or functionally equivalent duplicate exists."
        decision = "STOP"
    elif max_score >= 71:
        assessment = "AT RISK: a near duplicate exists; repositioning required."
        decision = "REVISE"
    elif max_score >= 46:
        assessment = "PARTIAL OVERLAP: meaningful overlap exists; emphasize differentiation."
        decision = "REVISE"
    elif max_score >= 21:
        assessment = "ADJACENT: adjacent projects exist but no duplicate; proceed with documented differentiation."
        decision = "PROCEED"
    else:
        assessment = "ORIGINAL: no meaningful duplicate found."
        decision = "PROCEED"
    lines.append(f"- Max duplication risk score: {max_score}")
    lines.append(f"- Assessment: {assessment}")
    lines.append("")
    lines.append("## Recommended decision")
    lines.append("")
    lines.append(f"**{decision}**")
    lines.append("")
    lines.append("## Required changes to benchmark positioning")
    lines.append("")
    if decision == "STOP":
        lines.append("- Redesign or reposition the benchmark to avoid the duplicate.")
    elif decision == "REVISE":
        lines.append("- Add an explicit differentiation argument.")
        lines.append("- Emphasize the unique cost-per-correct-answer metric.")
        lines.append("- Emphasize the PyTorch-first evaluation engine.")
    else:
        lines.append("- None required. Document adjacency for transparency.")
    lines.append("")
    lines.append("## Whether the benchmark can proceed")
    lines.append("")
    if decision in ("PROCEED", "REVISE"):
        lines.append("The benchmark may proceed (with revision if indicated).")
    else:
        lines.append("The benchmark must be stopped and repositioned.")
    lines.append("")
    return "\n".join(lines)


report_text = build_report()
with open(REPORT_PATH, "w", encoding="utf-8") as fh:
    fh.write(report_text)
print("Report written to", REPORT_PATH)


## 14. Legitimacy QA update (honest)

Update the originality component of the legitimacy QA report based on the sweep
result. The score is never forced above 85. Q-001 is only marked resolved when
this notebook has actually run on Kaggle and produced outputs.


In [ ]:
def compute_originality_delta():
    max_score = max((r.get("duplication_risk_score", 0) for r in ranked), default=0)
    if max_score >= 91:
        return -6, "exact duplicate found"
    if max_score >= 71:
        return -4, "near duplicate found; repositioning required"
    if max_score >= 46:
        return -2, "meaningful overlap found"
    if max_score >= 21:
        return 0, "adjacent projects only; no duplicate"
    return 1, "no meaningful duplicate found"


def update_legitimacy_report():
    delta, reason = compute_originality_delta()
    # Base provisional originality score was 8/10.
    base = 8
    new_originality = max(0, min(10, base + delta))
    content = []
    content.append("# Legitimacy QA Report")
    content.append("")
    content.append("## Status: UPDATED BY NOTEBOOK 01 (Kaggle execution)")
    content.append("")
    content.append(f"**Sweep executed at (UTC):** {EXECUTED_AT}")
    content.append("")
    content.append(f"**Originality component:** {new_originality}/10 (base 8, delta {delta:+d}; {reason})")
    content.append("")
    content.append("## Duplication sweep outcome")
    content.append("")
    content.append(f"- Max duplication risk score: {max((r.get('duplication_risk_score', 0) for r in ranked), default=0)}")
    content.append(f"- High risk (>=71): {len(high_risk)}")
    content.append(f"- Meaningful overlap (46-70): {len(meaningful)}")
    content.append(f"- Adjacent (21-45): {len(adjacent)}")
    content.append(f"- Distinct (0-20): {len(distinct)}")
    content.append("")
    content.append("## Provisional dimension scores")
    content.append("")
    content.append("| Dimension | Score (/10) | Rationale |")
    content.append("|-----------|-------------|-----------|")
    content.append(f"| originality | {new_originality} | {reason} |")
    content.append("| practical usefulness | 9 | Directly useful for model selection. |")
    content.append("| benchmark clarity | 9 | Clear metric, tasks, validators. |")
    content.append("| automatic evaluability | 9 | All v0.1 tasks auto-scorable. |")
    content.append("| PyTorch centrality | 8 | Evaluator, baselines, metrics use PyTorch. |")
    content.append("| Hugging Face fit | 8 | Dataset, Space, model repos planned. |")
    content.append("| Kaggle reproducibility | 9 | Kaggle-first notebooks and configs. |")
    content.append("| cost discipline | 9 | Hard caps, guards, dry-run default. |")
    content.append("| model pool validity | 6 | Prices not yet verified; pending. |")
    content.append("| release readiness | 7 | Skeletons ready; pending price verification. |")
    content.append("")
    total = new_originality + 9 + 9 + 9 + 8 + 8 + 9 + 9 + 6 + 7
    content.append(f"**Total:** {total}/100")
    content.append("")
    if total >= 85:
        content.append("Verdict: ACCEPTED (>=85).")
    else:
        content.append("Verdict: BELOW 85. Not accepted. Pending OpenRouter price verification (notebook 02) and, if needed, repositioning.")
    content.append("")
    with open(LEGITIMACY_PATH, "w", encoding="utf-8") as fh:
        fh.write("\n".join(content))
    print("Legitimacy report updated. Total:", total)


update_legitimacy_report()


## 15. Memory and QA update

Update memory and QA files after the sweep has run on Kaggle. Q-001 is marked
resolved only because this cell executes after a real Kaggle run produced
outputs above.


In [ ]:
def update_memory_and_qa():
    max_score = max((r.get("duplication_risk_score", 0) for r in ranked), default=0)

    # research_findings.md
    rf = []
    rf.append("# Research Findings")
    rf.append("")
    rf.append(f"| ID | Category | Source | Summary | Status |")
    rf.append(f"|----|----------|--------|---------|--------|")
    rf.append(f"| F-001 | theme | internal | SLM Efficiency Frontier selected as benchmark theme. | confirmed |")
    rf.append(f"| F-002 | hf_duplication | Kaggle notebook 01 | HF duplication sweep executed at {EXECUTED_AT}. Max risk score {max_score}. | confirmed |")
    rf.append(f"| F-003 | openrouter_models | pending | Model pool prices not yet verified on Kaggle. | pending |")
    rf.append(f"| F-004 | literature | pending | Literature sweep not yet executed. | pending |")
    with open(RESEARCH_FINDINGS_PATH, "w", encoding="utf-8") as fh:
        fh.write("\n".join(rf) + "\n")

    # qa_findings.md (Q-001 resolved only after real execution)
    qa = []
    qa.append("# QA Findings")
    qa.append("")
    qa.append("| ID | Area | Severity | Description | Status |")
    qa.append("|----|------|----------|-------------|--------|")
    qa.append(f"| Q-001 | research | BLOCKER | HF duplication sweep executed on Kaggle at {EXECUTED_AT}; max risk score {max_score}. | resolved |")
    qa.append("| Q-002 | budget | MAJOR | Model pool prices are placeholders; pending Kaggle notebook 02. | open |")
    qa.append("| Q-003 | metrics | MINOR | Determinism tests not yet run (Kaggle-only). | open |")
    qa.append("| Q-015 | runtime | MAJOR | Real OpenRouter execution (_real_call) not implemented. | open |")
    with open(QA_FINDINGS_PATH, "w", encoding="utf-8") as fh:
        fh.write("\n".join(qa) + "\n")

    # change_log.md (append)
    entry = (
        f"\n## {EXECUTED_AT} (Kaggle notebook 01 execution)\n"
        "- Executed HF duplication sweep on Kaggle.\n"
        f"- Max duplication risk score: {max_score}.\n"
        f"- High risk: {len(high_risk)}; meaningful: {len(meaningful)}; "
        f"adjacent: {len(adjacent)}; distinct: {len(distinct)}.\n"
        "- Updated hf_duplication_sweep.md, hf_duplication_sweep_results.json, "
        "legitimacy_qa_report.md, research_snapshot.md, research_findings.md, "
        "qa_findings.md.\n"
        "- Marked Q-001 resolved after real Kaggle execution.\n"
    )
    with open(CHANGE_LOG_PATH, "a", encoding="utf-8") as fh:
        fh.write(entry)

    # open_questions.md (keep price verification pending)
    oq = []
    oq.append("# Open Questions")
    oq.append("")
    oq.append(f"- OQ-001: HF duplication sweep executed on Kaggle at {EXECUTED_AT}. Max risk score {max_score}.")
    oq.append("- OQ-002: OpenRouter price verification still pending (Kaggle notebook 02).")
    oq.append("- OQ-003: Final public/private split ratio pending dataset curator design.")
    oq.append("- OQ-004: PyTorch baseline architectures for calibration pending baseline engineer.")
    with open(OPEN_QUESTIONS_PATH, "w", encoding="utf-8") as fh:
        fh.write("\n".join(oq) + "\n")

    print("Memory and QA files updated.")


update_memory_and_qa()


## 16. Review research snapshot update


In [ ]:
def update_research_snapshot():
    max_score = max((r.get("duplication_risk_score", 0) for r in ranked), default=0)
    snap = []
    snap.append("# Research Snapshot")
    snap.append("")
    snap.append(f"**Updated by Kaggle notebook 01 at (UTC):** {EXECUTED_AT}")
    snap.append("")
    snap.append("## Theme")
    snap.append("SLM Efficiency Frontier (selected). v0.1 is strictly English-only.")
    snap.append("")
    snap.append("## HF duplication sweep")
    snap.append(f"- Executed: yes (Kaggle, {EXECUTED_AT}).")
    snap.append(f"- Max duplication risk score: {max_score}.")
    snap.append(f"- High risk (>=71): {len(high_risk)}.")
    snap.append(f"- Meaningful overlap (46-70): {len(meaningful)}.")
    snap.append(f"- Adjacent (21-45): {len(adjacent)}.")
    snap.append(f"- Distinct (0-20): {len(distinct)}.")
    snap.append("")
    snap.append("## Pending")
    snap.append("- OpenRouter price verification (notebook 02).")
    snap.append("- Real OpenRouter execution implementation (notebook 04).")
    snap.append("- Final runtime QA (notebook 05).")
    with open(SNAPSHOT_PATH, "w", encoding="utf-8") as fh:
        fh.write("\n".join(snap) + "\n")
    print("Research snapshot updated:", SNAPSHOT_PATH)


update_research_snapshot()


## Next steps

1. Inspect `artifacts/research/hf_duplication_sweep.md` and the highest-risk rows.
2. If a near duplicate exists, reposition the benchmark and document the
   differentiation argument.
3. Proceed to notebook 02 (OpenRouter price verification) only after the
   duplication sweep decision is PROCEED or REVISE-with-acceptable-plan.
4. Never interpret any dry-run result as model performance.

This notebook does not call OpenRouter, does not perform paid inference, does
not train models, and does not download large artifacts. It is metadata-level
research only.
